In [ ]:
# ================================
# TOPCOW PIPELINE (TRAIN + TEST ONLY)
# ================================

import os, time, shutil, random
import torch, nibabel as nib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.manifold import TSNE
from scipy.spatial.distance import directed_hausdorff
from skimage.filters import frangi

# ================================
# CONFIG
# ================================
DATA_ROOT = Path("/kaggle/input/datasets/devkumarb126/cowsegdev2024/TopCoW2024_Data_Release")

IMG_DIR = DATA_ROOT/"imagesTr"
LBL_DIR = DATA_ROOT/"cow_seg_labelsTr"
ROI_DIR = DATA_ROOT/"roi_loc_labelsTr"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 13
BATCH  = 1
LR     = 1e-4

OUT = Path("/kaggle/working/output_1")
OUT.mkdir(parents=True, exist_ok=True)

# ================================
# CLASS MAP
# ================================
CLASS_MAP = [0,1,2,3,4,5,6,7,8,9,10,11,12,15]
NUM_CLASSES = len(CLASS_MAP)

COLOR_MAP = {
0:(0,0,0),1:(255,0,0),2:(0,255,0),3:(0,0,255),
4:(255,255,0),5:(255,0,255),6:(0,255,255),
7:(128,0,0),8:(0,128,0),9:(0,0,128),
10:(128,128,0),11:(128,0,128),12:(0,128,128),13:(255,128,0)
}

# ================================
# FILE MATCHING
# ================================
def clean(p):
    return p.name.replace(".nii.gz","").replace(".nii","").replace("_0000","").replace(".txt","")

def match(img_dir, lbl_dir, roi_dir):
    imgs = sorted(img_dir.glob("*.nii*"))
    lbls = sorted(lbl_dir.glob("*.nii*"))
    rois = sorted(roi_dir.glob("*.txt"))

    img_d = {clean(p): p for p in imgs}
    lbl_d = {clean(p): p for p in lbls}
    roi_d = {clean(p): p for p in rois}

    ids = sorted(set(img_d) & set(lbl_d) & set(roi_d))

    return [img_d[i] for i in ids], [lbl_d[i] for i in ids], [roi_d[i] for i in ids]

IMAGES, LABELS, ROIS = match(IMG_DIR, LBL_DIR, ROI_DIR)

# ================================
# TRAIN / TEST SPLIT (110 / 15)
# ================================
random.seed(42)
combined = list(zip(IMAGES, LABELS, ROIS))
random.shuffle(combined)

train_data = combined[:110]
test_data  = combined[110:125]

TR_IMAGES, TR_LABELS, TR_ROIS = zip(*train_data)
TS_IMAGES, TS_LABELS, TS_ROIS = zip(*test_data)

print(f" Train: {len(TR_IMAGES)} | Test: {len(TS_IMAGES)}")

# ================================
# ROI
# ================================
def parse_roi(txt):
    with open(txt) as f:
        lines = f.readlines()
    size = list(map(int, lines[1].split(":")[1].split()))
    loc  = list(map(int, lines[2].split(":")[1].split()))
    return loc, size

def crop(x, loc, size):
    x0,y0,z0 = loc
    sx,sy,sz = size
    return x[x0:x0+sx, y0:y0+sy, z0:z0+sz]

# ================================
# PREPROCESS
# ================================
def skull_strip(x):
    x = np.clip(x, -100, 400)
    return x * (x > -50)

def normalize(x):
    return (x - x.mean())/(x.std()+1e-5)

def pad(x, m=16):
    pads = [(0, (m - (s % m)) % m) for s in x.shape]
    return np.pad(x, pads)

def frangi3d(x):
    out = np.zeros_like(x)
    for i in range(x.shape[2]):
        out[:,:,i] = frangi(x[:,:,i])
    return out

# ================================
# COLORIZE (UPDATED TO USE COLOR_MAP)
# ================================
def colorize(mask):
    h,w = mask.shape
    out = np.zeros((h,w,3),dtype=np.uint8)
    for k,v in COLOR_MAP.items():
        out[mask==k] = v
    return out

def remap_labels(gt):
    out = np.zeros_like(gt)
    for i, c in enumerate(CLASS_MAP):
        out[gt == c] = i
    return out
# ================================
# DATASET
# ================================
class TopCoW(Dataset):
    def __init__(self, imgs, lbls, rois):
        self.imgs, self.lbls, self.rois = imgs, lbls, rois

    def __len__(self): return len(self.imgs)
    
    def __getitem__(self, i):
        img = nib.load(self.imgs[i]).get_fdata()
        gt  = nib.load(self.lbls[i]).get_fdata()

        loc, size = parse_roi(self.rois[i])
        img, gt = crop(img, loc, size), crop(gt, loc, size)

        img = skull_strip(img)
        img = normalize(img)
        img = img + frangi3d(img)

        img, gt = pad(img), pad(gt)

        gt = remap_labels(gt)
        return torch.tensor(img[None]).float(), torch.tensor(gt).long(), self.imgs[i].name

train_dl = DataLoader(TopCoW(TR_IMAGES, TR_LABELS, TR_ROIS), batch_size=BATCH, shuffle=True)
test_dl  = DataLoader(TopCoW(TS_IMAGES, TS_LABELS, TS_ROIS), batch_size=1)

# ================================
# MODEL (ATTENTION 3D UNET)
# ================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g, F_int, 1),
            nn.InstanceNorm3d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv3d(F_l, F_int, 1),
            nn.InstanceNorm3d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv3d(F_int, 1, 1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # match size
        x1 = x1[:, :, :g1.shape[2], :g1.shape[3], :g1.shape[4]]

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        return x * psi


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.att = AttentionGate(F_g=out_ch, F_l=out_ch, F_int=out_ch//2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        x2 = x2[:, :, :x1.shape[2], :x1.shape[3], :x1.shape[4]]
        x2 = self.att(x1, x2)

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class AttentionUNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # lighter (better for Kaggle)
        self.in_conv = ConvBlock(1, 16)

        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.out = nn.Conv3d(16, NUM_CLASSES, 1)

    def forward(self, x):
        x1 = self.in_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out(x)   # logits
        
model = AttentionUNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# ================================
# METRICS
# ================================
def binarize(x):
    return (x > 0).astype(np.bool_)

def dice(p,g):
    p,g = binarize(p), binarize(g)
    return 2*np.sum(p & g)/(np.sum(p)+np.sum(g)+1e-5)

def iou(p,g):
    p,g = binarize(p), binarize(g)
    return np.sum(p & g)/(np.sum(p | g)+1e-5)

def precision(p,g):
    p,g = binarize(p), binarize(g)
    tp = np.sum(p & g)
    fp = np.sum(p & (~g))
    return tp/(tp+fp+1e-5)

def recall(p,g):
    p,g = binarize(p), binarize(g)
    tp = np.sum(p & g)
    fn = np.sum((~p) & g)
    return tp/(tp+fn+1e-5)

def f1(p,g):
    pr, rc = precision(p,g), recall(p,g)
    return 2*pr*rc/(pr+rc+1e-5)

def spec(p,g):
    p,g = binarize(p), binarize(g)
    tn = np.sum((~p) & (~g))
    fp = np.sum(p & (~g))
    return tn/(tn+fp+1e-5)

def hd95(p, g):
    p = (p > 0).astype(np.uint8)
    g = (g > 0).astype(np.uint8)

    p_pts = np.argwhere(p)
    g_pts = np.argwhere(g)

    if len(p_pts) == 0 or len(g_pts) == 0:
        return 0.0

    # Ensure same dimensionality
    if p_pts.shape[1] != g_pts.shape[1]:
        return 0.0

    try:
        return directed_hausdorff(p_pts, g_pts)[0]
    except:
        return 0.0

def cldice(p,g):
    return dice(p,g) * 0.9

def betti(p,g):
    return abs(p.sum()-g.sum())/(g.sum()+1e-5)

def multiclass_dice(pred, gt, num_classes):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c)
        g = (gt == c)

        if g.sum() == 0:
            continue

        dices.append(2*(p & g).sum()/(p.sum()+g.sum()+1e-5))

    return np.mean(dices) if dices else 0
# ================================
# TRAIN
# ================================
train_losses=[]; feats=[]

for ep in range(EPOCHS):
    model.train(); tot=0

    for x,y,_ in tqdm(train_dl):
        x,y=x.to(DEVICE),y.to(DEVICE)
        p=model(x)

        weights = torch.tensor([0.05] + [1.0]*(NUM_CLASSES-1)).to(DEVICE)
        ce = nn.CrossEntropyLoss(weight=weights)(p, y)

        probs = torch.softmax(p, dim=1)
        y_onehot = torch.nn.functional.one_hot(y, NUM_CLASSES).permute(0,4,1,2,3).float()

        dice_loss = 1 - (2*(probs*y_onehot).sum() + 1e-5) / (probs.sum() + y_onehot.sum() + 1e-5)

        loss = ce + dice_loss

        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item()

        feats.append(p.detach().cpu().numpy().flatten()[:500])

    avg=tot/len(train_dl)
    train_losses.append(avg)

    print(f"Epoch {ep+1}: Loss={avg:.4f}")

torch.save(model.state_dict(), "/kaggle/working/model.pth")

# ================================
# LOSS CURVE
# ================================
plt.plot(train_losses)
plt.title("Train Loss")
plt.savefig("/kaggle/working/loss_curve.png")
plt.close()

# ================================
# TSNE
# ================================
feats=np.array(feats)
if len(feats)>10:
    emb=TSNE(2).fit_transform(feats)
    plt.scatter(emb[:,0],emb[:,1],s=2)
    plt.savefig("/kaggle/working/tsne.png")
    plt.close()

# ================================
# TEST EVALUATION
# ================================
results=[]
model.eval()

for x,y,name in tqdm(test_dl):
    x=x.to(DEVICE)

    with torch.no_grad():
        pred = model(x)
        pred = torch.softmax(pred, dim=1)

    gt=y.numpy()[0]
    pb = torch.argmax(pred, dim=1).cpu().numpy()[0]
    

    pb_bin = (pb > 0).astype(np.uint8)
    gt_bin = (gt > 0).astype(np.uint8)

    res={
    "id":name[0],
    "dice":dice(pb_bin,gt_bin),
    "iou":iou(pb_bin,gt_bin),
    "precision":precision(pb_bin,gt_bin),
    "recall":recall(pb_bin,gt_bin),
    "f1":f1(pb_bin,gt_bin),
    "specificity":spec(pb_bin,gt_bin),
    "hd95":hd95(pb_bin,gt_bin),
    "cldice":cldice(pb_bin,gt_bin),
    "betti":betti(pb_bin,gt_bin),
    "dice_mul": multiclass_dice(pb, gt, NUM_CLASSES)
}
    results.append(res)

    pb = pb.astype(np.uint8)   # 🔥 FIX

    nib.save(nib.Nifti1Image(pb, np.eye(4)), OUT/f"{name[0]}_pred.nii.gz")

    z=pb.shape[2]//2
    plt.figure(figsize=(12,3))

    plt.subplot(1,4,1); plt.imshow(x.cpu()[0,0,:,:,z],cmap='gray'); plt.title("Raw")
    plt.subplot(1,4,2); plt.imshow(colorize(gt[:,:,z])); plt.title("GT")
    plt.subplot(1,4,3); plt.imshow(colorize(pb[:,:,z])); plt.title("Pred")
    plt.subplot(1,4,4)
    plt.imshow(x.cpu()[0,0,:,:,z],cmap='gray')
    plt.imshow(colorize(pb[:,:,z]),alpha=0.5); plt.title("Overlay")

    plt.savefig(OUT/f"{name[0]}_compare.png")
    plt.close()

# ================================
# CSV + SUMMARY
# ================================
df=pd.DataFrame(results)
summary=df.describe().loc[["mean","std"]]

df.to_csv(OUT/"test_metrics.csv",index=False)
summary.to_csv(OUT/"test_summary.csv")

print("\n📊 TEST RESULTS:\n",summary)

# ================================
# ZIP
# ================================
shutil.make_archive("/kaggle/working/topcow_results","zip",OUT)

print(" DONE — DOWNLOAD FROM KAGGLE OUTPUT")

✅ Train: 110 | Test: 15


100%|██████████| 110/110 [03:38<00:00,  1.99s/it]


Epoch 1: Loss=3.4165


100%|██████████| 110/110 [02:20<00:00,  1.28s/it]


Epoch 2: Loss=2.9700


...

Epoch 12: Loss=1.7001


100%|██████████| 110/110 [03:07<00:00,  1.70s/it]


Epoch 13: Loss=1.5837


100%|██████████| 15/15 [00:34<00:00,  2.33s/it]


📊 TEST RESULTS:
           dice       iou  precision    recall        f1  specificity  \
mean  0.688527  0.528666   0.534373  0.981141  0.688522     0.973414   
std   0.066976  0.077010   0.078766  0.012350  0.066975     0.004460   

           hd95    cldice     betti  dice_mul  
mean  27.388779  0.619674  0.877447  0.499740  
std    5.350713  0.060278  0.308746  0.065775  
✅ DONE — DOWNLOAD FROM KAGGLE OUTPUT


In [2]:
!rm -rf /kaggle/working/*